<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook follows the gnomAD HGDP+1kGP PCA tutorial to build a global reference projection. It QC-filters the HGDP+TGP dense MatrixTable, LD-prunes, removes related samples, computes global and subcontinental PCs, and plots continent and subcontinent structure.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

# Project samples into the HGDP+TGP dataset

## Imports

In [ ]:
import os
import sys
import pandas as pd
import re

import math
import datetime
from dateutil import parser
from time import sleep
from typing import List
import pprint

from google.cloud import storage
my_storage_client = storage.Client()

from tqdm.notebook import tqdm

In [ ]:
import hail as hl

# Functions from gnomAD library to apply genotype filters and project related samples  
from gnomad.utils.filtering import filter_to_adj
from gnomad.sample_qc.ancestry import pc_project

# For plotting in Hail
from hail.ggplot import *
import plotly

from hail.plot import output_notebook, show
from bokeh.resources import INLINE
import bokeh.io

bokeh.io.output_notebook(INLINE)

In [ ]:
# Workspace details Terra sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']       # the workspace bucket
workspace = os.environ['WORKSPACE_NAME']      # workspace name
namespace = os.environ['WORKSPACE_NAMESPACE'] # workspace namespace

## Set paths

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# The 1,027-sample GLnexus joint-call MatrixTable (GRCh38).
grch38_mt_path = f'{AOU_LR_DATA}/outputs/GRCh38/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1027/cohort_for_GLNexus_2023Q1_1027.mt'

## Initialize Hail

In [ ]:
hl.init(default_reference='GRCh38', idempotent=True)

In [ ]:
grch38_mt = hl.read_matrix_table(grch38_mt_path)

In [ ]:
# This notebook follows gnomAD's HGDP+1kGP PCA tutorial to build the reference panel: sample, variant,
# and genotype QC, LD pruning, removal of related samples, then global and per-region PCA.
# Intermediate results are read from gnomAD's public copies where the tutorial provides them.
# Path for HGDP+1kGP dataset prior to applying gnomAD QC filters
pre_qc_path = 'gs://gcp-public-data--gnomad/release/3.1.2/mt/genomes/gnomad.genomes.v3.1.2.hgdp_1kg_subset_dense.mt'

# Path for gnomAD's HGDP+1kGP metadata for plotting 
metadata_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/metadata_and_qc/gnomad_meta_v1.tsv'

# Save the filtered and LD pruned mt as an intermediate file since LD pruning takes a while to rerun
ld_pruned_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/pca_preprocessing/ld_pruned.mt'

# Hail table of related sample IDs for separating unrelateds and relateds for PCA run 
related_sample_ids_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/pca_preprocessing/related_sample_ids.ht'

# Path for with-outliers PCA results - global & subcontinental PCA 
pc_scores_with_outliers_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/pca/pc_scores_with_outliers/'

# PCA outliers file 
outliers_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/pca/pca_outliers.txt'

# Path for without-outliers PCA results - global & subcontinental PCA 
pc_scores_without_outliers_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/pca/pc_scores_without_outliers/'

# Paths for unrelated and related datasets without outliers   
unrelateds_mt_without_outliers_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/pca_results/unrelateds_without_outliers.mt'
relateds_mt_without_outliers_path = 'gs://gcp-public-data--gnomad/release/3.1/secondary_analyses/hgdp_1kg/pca_results/relateds_without_outliers.mt' 

In [ ]:
# Set up function to apply gnomAD's sample, variant and genotype QC filters

def run_qc(mt):
    
    ## Apply sample QC filters to dataset 
    # This filters to only samples that passed gnomAD's sample QC hard filters  
    mt = mt.filter_cols(~mt.gnomad_sample_filters.hard_filtered) # removed 31 samples
    
    ## Apply variant QC filters to dataset
    # This subsets to only PASS variants - those which passed gnomAD's variant QC
    # PASS variants have an entry in the filters field 
    mt = mt.filter_rows(hl.len(mt.filters) != 0, keep=False)

    ## Apply genotype QC filters to the dataset
    # This is done using a function imported from gnomAD and is the last step in the QC process
    mt = filter_to_adj(mt)

    return mt

In [ ]:
# Read in the HGDP+1kGP pre-QC mt
pre_qc_mt = hl.read_matrix_table(pre_qc_path)

# Run QC 
post_qc_mt = run_qc(pre_qc_mt)

# Validity check: number of variants and samples after applying QC filters
# Took ~13min to print 
print('Num of SNVs and samples after applying QC filters = ' + str(post_qc_mt.count()))

In [ ]:
# Run Hail's common variant statistics (QC metrics) 
var_qc_mt = hl.variant_qc(post_qc_mt) 

# Filter to variants with AF between 0.05 & 0.95, and call rate greater than 0.999    
filtered_mt = var_qc_mt.filter_rows(((var_qc_mt.variant_qc.AF[0] > 0.05) & (var_qc_mt.variant_qc.AF[1] > 0.05)) &
                                 ((var_qc_mt.variant_qc.AF[0] < 0.95) & (var_qc_mt.variant_qc.AF[1] < 0.95)) &
                                 (var_qc_mt.variant_qc.call_rate > 0.999))
# Took ~13min to print 
print(f'Num of variants after filtering = {filtered_mt.count_rows()}')

In [ ]:
# Remove correlated variants 
# Took 1hr & 15min to run 
pruned_ht = hl.ld_prune(filtered_mt.GT, r2=0.1, bp_window_size=500000)

In [ ]:
filtered_pruned_mt = filtered_mt.filter_rows(hl.is_defined(pruned_ht[filtered_mt.row_key]))

In [ ]:
# Took ~13min to print 
print(f'Num of variants after LD pruning = {filtered_pruned_mt.count_rows()}')

In [ ]:
# gnomAD's saved LD-pruned MatrixTable, used in place of the one computed above.
# Read the intermediate file back in for subsequent analyses
filtered_pruned_mt = hl.read_matrix_table(ld_pruned_path) 

In [ ]:
# Compute kinship statistic (took ~30min to run)
relatedness_ht = hl.pc_relate(
    filtered_pruned_mt.GT, 
    min_individual_maf=0.05, 
    min_kinship=0.05, 
    statistics='kin', 
    k=20).key_by() 

In [ ]:
# Identify closely related individuals in pairs (list of sample IDs) 
related_sample_ids = hl.maximal_independent_set(relatedness_ht.i, relatedness_ht.j, False) # 721 samples

In [ ]:
# Read the list of related sample IDs back in
related_sample_ids = hl.read_table(related_sample_ids_path)

# Subset the filtered and pruned mt to unrelated samples only 
# Sample IDs that are NOT present in the list of related individuals  
unrelateds_mt_preoutlier = filtered_pruned_mt.filter_cols(hl.is_defined(related_sample_ids[filtered_pruned_mt.col_key]), keep=False) 

# Do the same as above but this time subset to related samples only 
# Sample IDs that are present in the list of related individuals    
relateds_mt_preoutlier = filtered_pruned_mt.filter_cols(hl.is_defined(related_sample_ids[filtered_pruned_mt.col_key]), keep=True) 

In [ ]:
def run_pca(mt: hl.MatrixTable):
    """
    Runs PCA on a dataset
    :param mt: dataset to run PCA on
    :return: loadings and pc scores of unrelated samples 
    """
    pca_evals, pca_scores, pca_loadings = hl.hwe_normalized_pca(mt.GT, k=20, compute_loadings=True)
    pca_mt = mt.annotate_rows(pca_af=hl.agg.mean(mt.GT.n_alt_alleles()) / 2)
    pca_loadings = pca_loadings.annotate(pca_af=pca_mt.rows()[pca_loadings.key].pca_af)
    pca_scores = pca_scores.transmute(**{f'PC{i}': pca_scores.scores[i - 1] for i in range(1, 21)})
    
    return pca_loadings, pca_scores 

In [ ]:
def project_individuals(project_mt, pca_loadings, unrel_scores, out_path: str, reg_name:str, outlier_status:str):
    """
    Project samples into predefined PCA space
    :param project_mt: matrix table of related samples to project 
    :param pca_loadings: existing PCA space of unrelated samples 
    :param unrel_scores: unrelated samples' PC scores
    :param out_path: path for where to save PCA projection outputs
    :param reg_name: region name for saving output purposes
    :param outlier_status: is the dataset with or without outliers? 
    """
    ht_projections = pc_project(project_mt, pca_loadings)  
    ht_projections = ht_projections.transmute(**{f'PC{i}': ht_projections.scores[i - 1] for i in range(1, 21)}) 
    scores = unrel_scores.union(ht_projections) # combine the pc scores from both the unrelateds and relateds 
    scores.export(out_path + reg_name + '_scores_' + outlier_status + '.txt.bgz') # write output for plotting    

In [ ]:
def plot_global_pca(scores_path, outlier_status):
    """
    :param scores_path: general path where PC score files are located 
    :param outlier_status: are outlier samples present in the scores file (use "with_outliers" OR "without_outliers")
    """
    # Dictionary mapping colors to region names 
    cont_colors = {'AMR':"#E41A1C",
               'AFR':"#984EA3", 
               'OCE':"#999999",
               'CSA':"#FF7F00",
               'EAS':"#4DAF4A", 
               'EUR':"#377EB8", 
               'MID':"#A65628" }
    
    # Import global PC scores table 
    global_scores = hl.import_table(scores_path + 'GLOBAL_scores_' + outlier_status + '.txt.bgz', impute = True)

    # CHMI_CHMI3_WGS2 is a sample added by gnomAD for QC purposes and thus doesn't have metadata information 
    # To avoid a "None" error, we have to remove it before plotting 
    # It is treated as a PCA outlier in section 7 below 
    global_scores = global_scores.filter(global_scores.s == 'CHMI_CHMI3_WGS2', keep = False)

    # Add information from the metadata for plotting purposes 
    global_scores = global_scores.annotate(
        global_pop = metadata[global_scores.s]['hgdp_tgp_meta.Genetic.region'], 
        subpop = metadata[global_scores.s]['hgdp_tgp_meta.Population'],
        global_color = metadata[global_scores.s]['hgdp_tgp_meta.Continent.colors'],
        subpop_color = metadata[global_scores.s]['hgdp_tgp_meta.Pop.colors'],
        subpop_shapes = metadata[global_scores.s]['hgdp_tgp_meta.Pop.shapes'],
        proj_title = metadata[global_scores.s]['hgdp_tgp_meta.Project'])

    # Make plot
    # Only plotting PC1 vs PC2 here but you can change the PC values OR make a for loop to plot the rest of the PCs
    p = ggplot(global_scores, aes(x = global_scores.PC1, y = global_scores.PC2))+ \
        geom_point(aes(color = global_scores.global_pop,
                       shape = global_scores.proj_title),
                       size = 3, alpha = .5) +\
        xlab("PC1") + \
        ylab("PC2") + \
        ggtitle("Global PCA " + outlier_status.replace('_', ' '))+\
        labs(shape = 'Project', color = 'Population') +\
        scale_color_manual(values=cont_colors)

    return p 

In [ ]:
def plot_subcont_pca(scores_path, outlier_status):
    """
    :param scores_path: general path where PC score files are located  
    :param outlier_status: are outlier samples present in the scores file (use "with_outliers" OR "without_outliers")   
    """
    # Initialize a dictionary to save the subcontinental PCA plots by their respective regions 
    pca_plots = {}

    # Loop through each subcontinental region 
    regions = ['AFR', 'AMR', 'CSA', 'EAS', 'EUR', 'MID', 'OCE']
 
    for region in regions:
        # Import PC scores table 
        subcont_scores = hl.import_table(scores_path + region + '_scores_' + outlier_status + '.txt.bgz', impute = True)

        # Add information from the metadata for plotting purposes 
        subcont_scores = subcont_scores.annotate(
            global_pop = metadata[subcont_scores.s]['hgdp_tgp_meta.Genetic.region'], 
            subpop = metadata[subcont_scores.s]['hgdp_tgp_meta.Population'],
            global_color = metadata[subcont_scores.s]['hgdp_tgp_meta.Continent.colors'],
            subpop_color = metadata[subcont_scores.s]['hgdp_tgp_meta.Pop.colors'],
            subpop_shapes = metadata[subcont_scores.s]['hgdp_tgp_meta.Pop.shapes'],
            proj_title = metadata[subcont_scores.s]['hgdp_tgp_meta.Project'])

        # Make plot 
        # Only plotting PC1 vs PC2 here but you can change the PC values OR make a for loop to plot the rest of the PCs
        p = ggplot(subcont_scores, aes(x=subcont_scores.PC1, y=subcont_scores.PC2)) + \
            geom_point(aes(color = subcont_scores.subpop, 
                           shape = subcont_scores.proj_title),
                           size = 3, alpha = .3) +\
            xlab("PC1") + \
            ylab("PC2") + \
            ggtitle(region + " PCA " + outlier_status.replace('_', ' '))+\
            labs(shape = 'Project', color = 'Population')

        # Add plot to dictionary with the region name as its key 
        pca_plots[region] = p

    return pca_plots

In [ ]:
# Region and population labels for the plots.
# Read in gnomAD's HGDP+1kGP metadata for plot annotation
metadata = hl.import_table(metadata_path, impute = True, key = 's')


In [ ]:
# Calculate PC scores 

# This block took 23min to run 

# Dictionaries to hold unrelateds' PCA loadings and scores
loadings_dict = {}
unrel_scores_dict = {}

# Run PCA on unrelated samples as a whole
loadings_dict['GLOBAL'], unrel_scores_dict['GLOBAL'] = run_pca(unrelateds_mt_preoutlier)  

# Project related samples onto unrelated-samples' PC space 
project_individuals(relateds_mt_preoutlier, loadings_dict['GLOBAL'], unrel_scores_dict['GLOBAL'], pc_scores_with_outliers_path, 'GLOBAL', 'with_outliers')

In [ ]:
# Plot PCA 
global_with_outliers = plot_global_pca(pc_scores_with_outliers_path, "with_outliers") 

# Show PC1 Vs PC2
global_with_outliers.show()

In [ ]:
# Calculate PC scores 

# Run time breakdown for this cell is as follows:
# 1hr & 42 min for EAS, AMR, CSA
# 1hr & 23 min for EUR, AFR and OCE
# 34 min for MID

# Dictionaries to hold unrelateds' PCA loadings and scores
loadings_dict = {}
unrel_scores_dict = {}
regions = unrelateds_mt_preoutlier['hgdp_tgp_meta']['genetic_region'].collect() 
regions = list(dict.fromkeys(regions)) # convert into a list
# There are 7 regions: EUR, AFR, AMR, EAS, CSA, OCE, and MID

# For each region, run PCA on the unrelated samples 
for i in regions:  
    if i is not None: # exclude a none value
        # Filter the unrelateds per region
        subcont_unrelateds = unrelateds_mt_preoutlier.filter_cols(unrelateds_mt_preoutlier['hgdp_tgp_meta']['genetic_region'] == i) 

        # Run PCA
        loadings_dict[i], unrel_scores_dict[i] = run_pca(subcont_unrelateds)

        # Filter the related mt per region 
        subcont_relateds = relateds_mt_preoutlier.filter_cols(relateds_mt_preoutlier['hgdp_tgp_meta']['genetic_region'] == i)  

        # Project related samples onto unrelated-samples' PC space 
        project_individuals(subcont_relateds, loadings_dict[i], unrel_scores_dict[i], pc_scores_with_outliers_path, i, 'with_outliers')


In [ ]:
# Plot PCA
subcont_with_outliers = plot_subcont_pca(pc_scores_with_outliers_path, "with_outliers") 

# Show subcontinental PC1 Vs PC2 plots one by one 
for region in ['AFR', 'AMR', 'CSA', 'EAS', 'EUR', 'MID', 'OCE']:
    subcont_with_outliers[region].show()
    
# If you are only interested in one subcontinental region, you can do the following. Using AFR as an example:
subcont_with_outliers["AFR"].show()